In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
ADHD_file_path = r"C:\Users\tharu\Desktop\Starlight\FYP\Cross-Disorder_Multi-Label_Classification_Using_Transformer_and_Graph-Based_Fusion_for_Dyslexia_ADHD_and_Autism\Datasets\Genomic Data\ADHD2022\ADHD2022_iPSYCH_deCODE_PGC\ADHD2022.tsv"  
ASD_file_path = r"C:\Users\tharu\Desktop\Starlight\FYP\Cross-Disorder_Multi-Label_Classification_Using_Transformer_and_Graph-Based_Fusion_for_Dyslexia_ADHD_and_Autism\Datasets\Genomic Data\14671989\iPSYCH-PGC_ASD_Nov2017\ASD2019.tsv"
Dyslexia_file_path = r"C:\Users\tharu\Desktop\Starlight\FYP\Cross-Disorder_Multi-Label_Classification_Using_Transformer_and_Graph-Based_Fusion_for_Dyslexia_ADHD_and_Autism\Datasets\Genomic Data\Dyslexia_GWAS_Summary_Statistics_for_top_10K_SNPs\top_10K_variants.txt"


In [ ]:
ADHD_df = pd.read_csv(
    ADHD_file_path,
    delim_whitespace=True,
    low_memory=False
)

ASD_df = pd.read_csv(
    ASD_file_path,
    delim_whitespace=True,
    low_memory=False
)

dys_df = pd.read_csv(
    Dyslexia_file_path, 
    sep="\t", 
    low_memory=False)

C:\Users\tharu\AppData\Local\Temp\ipykernel_25728\1698581988.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  ADHD_df = pd.read_csv(
C:\Users\tharu\AppData\Local\Temp\ipykernel_25728\1698581988.py:7: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  ASD_df = pd.read_csv(


In [ ]:
ADHD_df.head()

In [ ]:
ASD_df.head()

In [ ]:
dys_df.head()

In [ ]:
n = 5  # number of values to inspect per column

for col in ADHD_df.columns:
    print(f"\nColumn: {col}")
    print(ADHD_df[col].head(n).to_list())


In [ ]:
n = 5  # number of values to inspect per column

for col in ASD_df.columns:
    print(f"\nColumn: {col}")
    print(ADHD_df[col].head(n).to_list())


In [ ]:
n = 5  # number of values to inspect per column

for col in dys_df.columns:
    print(f"\nColumn: {col}")
    print(dys_df[col].head(n).to_list())


In [ ]:
ADHD_df.info()

In [ ]:
ASD_df.info()

In [ ]:
dys_df.info()

 | Column           | Meaning                                                |
 | ---------------- | ------------------------------------------------------ |
 | **CHR**          | Chromosome number                                      |
 | **SNP**          | rsID                                                   |
 | **BP**           | Base-pair position (hg19 / hg38                        |
 | **A1**           | Effect allele                                          |
 | **A2**           | Other allele                                           |
 | **FRQ_A_38691**  | Effect allele frequency in cases                       |
 | **FRQ_U_186843** | Effect allele frequency in controls                    |
 | **INFO**         | Imputation quality (keep ≥ 0.8)                        |
 | **OR**           | Odds ratio (logistic GWAS)                             |
 | **SE**           | Standard error                                         |
 | **P**            | P-value                                                |
 | **Direction**    | Per-cohort effect direction                            |
 | **Nca**          | Number of cases                                        |
 | **Nco**          | Number of controls                                     |


### Rename Columns to match each dataset

In [ ]:
# Rename columns
ADHD_df = ADHD_df.rename(columns={
    "CHR": "CHR",
    "BP": "BP",
    "SNP": "SNP",
    "A1": "A1",
    "A2": "A2",
    "OR": "OR",
    "SE": "SE",
    "P": "P"
})

# Keep only required columns
ADHD_df = ADHD_df[["CHR", "BP", "SNP", "A1", "A2", "OR", "SE", "P"]]
ASD_df = ASD_df[["CHR", "BP", "SNP", "A1", "A2", "OR", "SE", "P"]]


In [ ]:
dys_df.rename(columns={
    "assay.name": "SNP",
    "scaffold": "CHR",
    "position": "BP",
    "effect_allele": "A1",
    "other_allele": "A2",
    "effect": "BETA",
    "stderr": "SE",
    "pvalue": "P",
    "avg.rsqr": "INFO"
}, inplace=True)

def chr_to_int(chr_val):
    chr_val = str(chr_val).replace("chr", "")  # remove "chr" prefix
    if chr_val.isdigit():
        return int(chr_val)
    elif chr_val.upper() == "X":
        return 23
    elif chr_val.upper() == "Y":
        return 24
    else:
        return np.nan  # for MT or unknown

dys_df["CHR"] = dys_df["CHR"].apply(chr_to_int)
dys_df = dys_df.dropna(subset=["CHR"])  # drop non-autosomes/unknown chromosomes
dys_df["CHR"] = dys_df["CHR"].astype(int)


In [ ]:
ADHD_df

In [ ]:
ASD_df

In [ ]:
dys_df

### Convert OR to BETA

In [ ]:
# For ADHD
if "OR" in ADHD_df.columns:
    ADHD_df["BETA"] = np.log(ADHD_df["OR"])
else:
    print("No OR column found in ADHD_df!")

# For ASD
if "OR" in ASD_df.columns:
    ASD_df["BETA"] = np.log(ASD_df["OR"])
else:
    print("No OR column found in ASD_df!")

In [ ]:
ADHD_df

In [ ]:
ASD_df

In [ ]:
dys_df

In [ ]:
assert np.isfinite(ADHD_df["BETA"]).all()
assert np.isfinite(ASD_df["BETA"]).all()
print("All BETA values are finite ")

### Converting the columns in dyslexia dataset is numeric

In [ ]:
dys_df["BETA"] = pd.to_numeric(dys_df["BETA"])
dys_df["SE"] = pd.to_numeric(dys_df["SE"])
dys_df["P"] = pd.to_numeric(dys_df["P"])
dys_df["BP"] = pd.to_numeric(dys_df["BP"])
dys_df["INFO"] = pd.to_numeric(dys_df["INFO"])

### P-value filtering

In [ ]:
P_THRESHOLD = 1e-5

ADHD_df = ADHD_df[ADHD_df["P"] <= P_THRESHOLD]
ADHD_df = ADHD_df.reset_index(drop=True)

ASD_df = ASD_df[ASD_df["P"] <= P_THRESHOLD]
ASD_df = ASD_df.reset_index(drop=True)

dys_df = dys_df[dys_df["P"] <= P_THRESHOLD]
dys_df = dys_df.reset_index(drop=True)

### Handling Missing Data

In [ ]:
ADHD_df = ADHD_df.dropna(subset=["BETA", "SE", "P"])
ASD_df = ASD_df.dropna(subset=["BETA", "SE", "P"])
dys_df = dys_df.dropna(subset=["BETA", "SE", "P"])

In [ ]:
print("ADHD_df shape: ", ADHD_df.shape)
print("ASD_df shape: ", ASD_df.shape)
print("Dyslexia_df shape: ", dys_df.shape)

### Exploratory Data Analysis

In [ ]:
# SNP Count per Chromosome

sns.countplot(data=ADHD_df, x="CHR")
plt.title("ADHD SNP count per chromosome")
plt.show()

sns.countplot(data=ASD_df, x="CHR")
plt.title("ASD SNP count per chromosome")
plt.show()

sns.countplot(data=dys_df, x="CHR")
plt.title("Dyslexia SNP count per chromosome")
plt.show()

In [ ]:
# P-value Distribution

plt.hist(-np.log10(ADHD_df['P']), bins=50)
plt.title("ADHD -log10(P) distribution")
plt.xlabel("-log10(P)")
plt.ylabel("Count")
plt.show()

plt.hist(-np.log10(ASD_df['P']), bins=50)
plt.title("ASD -log10(P) distribution")
plt.xlabel("-log10(P)")
plt.ylabel("Count")
plt.show()

plt.hist(-np.log10(dys_df['P']), bins=50)
plt.title("Dyslexia -log10(P) distribution")
plt.xlabel("-log10(P)")
plt.ylabel("Count")
plt.show()

In [ ]:
# Beta Distribution
sns.histplot(ADHD_df['BETA'], bins=50, kde=True)
plt.title("ADHD BETA distribution")
plt.show()

sns.histplot(ASD_df['BETA'], bins=50, kde=True)
plt.title("ASD BETA distribution")
plt.show()

sns.histplot(dys_df['BETA'], bins=50, kde=True)
plt.title("Dyslexia BETA distribution")
plt.show()

In [ ]:
#QQ-Plot (Compare expected vs observed p-values)

from statsmodels.graphics.gofplots import qqplot


qqplot(-np.log10(ADHD_df["P"]), line="45")
plt.title("QQ Plot - ADHD")
plt.show()

qqplot(-np.log10(ASD_df["P"]), line="45")
plt.title("QQ Plot - ASD")
plt.show()

qqplot(-np.log10(dys_df["P"]), line="45")
plt.title("QQ Plot - Dyslexia")
plt.show()

In [ ]:
#Manhattan Plot (For a visual overview of SNP significance across the genome)

ADHD_df['-log10P'] = -np.log10(ADHD_df['P'])
plt.scatter(ADHD_df['BP'], ADHD_df['-log10P'], c=ADHD_df['CHR'], cmap='tab20', s=10)
plt.xlabel("Base pair position")
plt.ylabel("-log10(P)")
plt.title("Manhattan plot - ADHD")
plt.show()

ASD_df['-log10P'] = -np.log10(ASD_df['P'])
plt.scatter(ASD_df['BP'], ASD_df['-log10P'], c=ASD_df['CHR'], cmap='tab20', s=10)
plt.xlabel("Base pair position")
plt.ylabel("-log10(P)")
plt.title("Manhattan plot - ASD")
plt.show()

dys_df['-log10P'] = -np.log10(dys_df['P'])
plt.scatter(dys_df['BP'], dys_df['-log10P'], c=dys_df['CHR'], cmap='tab20', s=10)
plt.xlabel("Base pair position")
plt.ylabel("-log10(P)")
plt.title("Manhattan plot - Dyslexia")
plt.show()

In [ ]:
# Pair Plots

adhd_sample = ADHD_df.sample(n=1000, random_state=42)

sns.pairplot(
    adhd_sample[['BETA', 'SE', 'P', 'CHR']],
    diag_kind='kde',
    corner=True
)

plt.suptitle("ADHD Pairplot (Sampled)", y=1.02)
plt.show()

asd_sample = ASD_df.sample(n=1000, random_state=42)

sns.pairplot(
    asd_sample[['BETA', 'SE', 'P', 'CHR']],
    diag_kind='kde',
    corner=True
)

plt.suptitle("ASD Pairplot (Sampled)", y=1.02)
plt.show()

dys_sample = dys_df.sample(n=1000, random_state=42)

sns.pairplot(
    dys_sample[['BETA', 'SE', 'P', 'CHR']],
    diag_kind='kde',
    corner=True
)

plt.suptitle("Dyslexia Pairplot (Sampled)", y=1.02)
plt.show()

In [ ]:
# Effect size vs significance

sns.scatterplot(
    x=ADHD_df['BETA'],
    y=-np.log10(ADHD_df['P']),
    s=10
)
plt.title("ADHD Effect size vs Significance")
plt.xlabel("BETA")
plt.ylabel("-log10(P)")
plt.show()

sns.scatterplot(
    x=ASD_df['BETA'],
    y=-np.log10(ADHD_df['P']),
    s=10
)
plt.title("ASD Effect size vs Significance")
plt.xlabel("BETA")
plt.ylabel("-log10(P)")
plt.show()

sns.scatterplot(
    x=dys_df['BETA'],
    y=-np.log10(ADHD_df['P']),
    s=10
)
plt.title("Dyslexia Effect size vs Significance")
plt.xlabel("BETA")
plt.ylabel("-log10(P)")
plt.show()


### Filter the top 10k SNP

In [ ]:
TOP_K = 10_000

ADHD_df = ADHD_df.sort_values("P").head(TOP_K)
ADHD_df = ADHD_df.reset_index(drop=True)

ASD_df = ASD_df.sort_values("P").head(TOP_K)
ASD_df = ASD_df.reset_index(drop=True)


In [ ]:
ADHD_df = ADHD_df.drop_duplicates(subset="SNP")
ASD_df = ASD_df.drop_duplicates(subset="SNP")
dys_df = dys_df.drop_duplicates(subset="SNP")

In [ ]:
print("ADHD_df shape: ", ADHD_df.shape)
print("ASD_df shape: ", ASD_df.shape)
print("Dyslexia_df shape: ", dys_df.shape)

### Standardizing and Scalling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
ADHD_df["BETA_scaled"] = scaler.fit_transform(ADHD_df[["BETA"]])
ASD_df["BETA_scaled"] = scaler.fit_transform(ASD_df[["BETA"]])
dys_df["BETA_scaled"] = scaler.fit_transform(dys_df[["BETA"]])

### Save to CSV

In [ ]:
ADHD_df.to_csv("ADHD_top10k_clean.csv", index=False)
ASD_df.to_csv("ASD_top10k_clean.csv", index=False)
dys_df.to_csv("Dyslexia_top10k_clean.csv", index=False)